# Explainable UID + QR Generator

This notebook is the **development version** of the UID/QR workflow.

It is designed for a real Excel file containing a continuous list of Existing IDs.

### Safety rules

- The notebook first identifies the **Existing ID** field.
- It identifies the first data row.
- It checks for repeated IDs **before generating anything**.
- It reads the previous master register, if present.
- Existing Existing ID → UID mappings are preserved.
- Only genuinely new IDs receive new UIDs.
- New UIDs are appended to the master register.
- QR codes are generated for the final UID set.
- A final integrity check confirms historical mappings were not changed.

For production, the secret key must be generated on the air-gapped Windows machine, not on the Mac.


## 1. Configure the files

The input workbook can have the `Existing ID` field anywhere within the first 20 rows and 30 columns.

The notebook does not assume a fixed number of records.

Set `INPUT_FILE` to your actual Excel file.


In [15]:
from pathlib import Path
import hashlib, hmac, base64, unicodedata

import openpyxl
from openpyxl import Workbook, load_workbook
from openpyxl.drawing.image import Image as XLImage
from openpyxl.styles import Font, PatternFill, Alignment
import qrcode

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "src" else Path.cwd()

INPUT_FILE = PROJECT_DIR / "input" / "sample.xlsx"
KEY_FILE = PROJECT_DIR / "test_key.bin"

OUTPUT_DIR = PROJECT_DIR / "output"
QR_DIR = PROJECT_DIR / "qr_codes"
MASTER_FILE = OUTPUT_DIR / "master_register.xlsx"
DISTRIBUTABLE_FILE = OUTPUT_DIR / "distributable_register.xlsx"

ID_HEADER = "Drone ID"
UID_HEADER = "UID"
QR_HEADER = "QR Code"

KEY_BYTES = 32
UID_BYTES = 12
UID_PREFIX = "UID-"

MAX_HEADER_SEARCH_ROWS = 20
MAX_HEADER_SEARCH_COLS = 30

print("Input:", INPUT_FILE)
print("Previous master:", MASTER_FILE)


Input: /Users/santhoshbalaiah/UID_QR_Generator/input/sample.xlsx
Previous master: /Users/santhoshbalaiah/UID_QR_Generator/output/master_register.xlsx


## 2. Inspect the workbook

We first confirm that the workbook exists and is a valid Excel workbook.

No UID generation happens in this stage.


In [16]:
if not INPUT_FILE.exists():
    raise FileNotFoundError(f"Input workbook not found: {INPUT_FILE}")

if INPUT_FILE.stat().st_size == 0:
    raise ValueError("Input workbook is empty.")

wb = load_workbook(INPUT_FILE, read_only=True, data_only=True)
print("Workbook:", INPUT_FILE.name)
print("Sheets:", wb.sheetnames)
for ws in wb.worksheets:
    row_count = 0
    column_count = 0
    for row in ws.iter_rows():
        row_count += 1
        column_count = max(column_count, len(row))
    print(f"  {ws.title}: {row_count} rows x {column_count} columns")
wb.close()


Workbook: sample.xlsx
Sheets: ['Sheet1']
  Sheet1: 123 rows x 21 columns


## 3. Identify the Existing ID field and first data row

The notebook searches for a header whose normalized value is exactly `Existing ID`.

Once found:

- **field** = the column containing Existing ID
- **start row** = header row + 1

This makes the process explainable instead of silently assuming column A/A1.


In [17]:
def norm(value):
    if value is None:
        return ""
    return " ".join(unicodedata.normalize("NFKC", str(value)).split()).casefold()

def find_existing_id_field(path):
    wb = load_workbook(path, read_only=True, data_only=True)
    matches = []
    for ws in wb.worksheets:
        for r, row in enumerate(ws.iter_rows(), start=1):
            if r > MAX_HEADER_SEARCH_ROWS:
                break
            for c, cell in enumerate(row, start=1):
                if c > MAX_HEADER_SEARCH_COLS:
                    break
                if norm(cell.value) == norm(ID_HEADER):
                    matches.append((ws.title, r, c))
    wb.close()

    if not matches:
        raise ValueError(f'Field "{ID_HEADER}" was not found.')
    if len(matches) > 1:
        raise ValueError(f'Multiple "{ID_HEADER}" fields found: {matches}')
    return matches[0]

sheet_name, header_row, id_col = find_existing_id_field(INPUT_FILE)

print("FIELD IDENTIFIED")
print("----------------")
print("Sheet:", sheet_name)
print("Field:", ID_HEADER)
print("Column:", id_col)
print("Header row:", header_row)
print("FIRST DATA ROW:", header_row + 1)


FIELD IDENTIFIED
----------------
Sheet: Sheet1
Field: Drone ID
Column: 2
Header row: 2
FIRST DATA ROW: 3


## 4. Read all continuous entries

All non-empty values below the identified header are read.

Blank cells are skipped.

The original value is retained for the master register; the normalized value is used for duplicate detection and deterministic UID generation.


In [18]:
def read_ids(path, sheet, header_row, id_col):
    wb = load_workbook(path, read_only=True, data_only=True)
    ws = wb[sheet]
    records = []
    for r, row in enumerate(ws.iter_rows(min_row=header_row + 1), start=header_row + 1):
        value = row[id_col - 1].value if len(row) >= id_col else None
        if value is None or str(value).strip() == "":
            continue
        records.append({
            "row": r,
            "original": value,
            "normalized": norm(value)
        })
    wb.close()
    return records

records = read_ids(INPUT_FILE, sheet_name, header_row, id_col)

print("ENTRIES IDENTIFIED")
print("------------------")
print("Total non-empty Drone IDs:", len(records))
for x in records[:10]:
    print(f"Excel row {x['row']}: {x['original']!r}")
if len(records) > 10:
    print("...")


ENTRIES IDENTIFIED
------------------
Total non-empty Drone IDs: 121
Excel row 3: 'DUMMY-1897-BED9-A8D3'
Excel row 4: 'DUMMY-1B07-B165-E4BC'
Excel row 5: 'DUMMY-DEE1-AAB5-F526'
Excel row 6: 'DUMMY-E88F-BEF8-3414'
Excel row 7: 'DUMMY-08BE-0F79-FE27'
Excel row 8: 'DUMMY-6DA8-DCC8-500E'
Excel row 9: 'DUMMY-2E3D-9084-E2D7'
Excel row 10: 'DUMMY-5B9C-AFD3-944D'
Excel row 11: 'DUMMY-5512-2BD7-911C'
Excel row 12: 'DUMMY-AC9C-48A9-2268'
...


## 5. Duplicate check — STOP before processing

This is a hard validation gate.

Examples such as:

- `DRONE-001`
- ` drone-001 `

are treated as the same ID after normalization.

If any repeated ID is found, the notebook stops here and does not write a new UID register.


In [19]:
from collections import defaultdict

seen = defaultdict(list)
for x in records:
    seen[x["normalized"]].append(x["row"])

duplicates = {k:v for k,v in seen.items() if k and len(v) > 1}

if duplicates:
    print("DUPLICATE IDs FOUND")
    for k, rows in duplicates.items():
        print(f"{k!r} -> Excel rows {rows}")
    raise ValueError("PROCESS STOPPED: repeated Drone IDs detected.")

if not records:
    raise ValueError("No Drone ID values were found.")

print("PASS: No repeated Drone IDs found.")
print("The UID generation stage may proceed.")


PASS: No repeated Drone IDs found.
The UID generation stage may proceed.


## 6. Load the secret key

The key is external to Excel.

For Mac development, use `test_key.bin`.

For production, generate the key on the secure/offline Windows machine.

The key itself is never printed.


In [20]:
if not KEY_FILE.exists():
    raise FileNotFoundError(f"Key not found: {KEY_FILE}")

key = KEY_FILE.read_bytes()
if len(key) != KEY_BYTES:
    raise ValueError(f"Expected a {KEY_BYTES}-byte key; found {len(key)} bytes.")

print("PASS: key loaded.")
print("Key length:", len(key), "bytes")


PASS: key loaded.
Key length: 32 bytes


## 7. Deterministic UID function

UID = HMAC-SHA256(secret key, normalized Existing ID).

The same Existing ID + same key always produces the same UID.

This is useful for repeatability, but the **previous master register remains authoritative** for historical mappings.


In [21]:
def make_uid(existing_id, key):
    normalized = norm(existing_id)
    digest = hmac.new(key, normalized.encode("utf-8"), hashlib.sha256).digest()
    encoded = base64.b32encode(digest[:UID_BYTES]).decode("ascii").rstrip("=")
    return UID_PREFIX + "-".join(encoded[i:i+4] for i in range(0, len(encoded), 4))

print("Example:")
print(records[0]["original"], "->", make_uid(records[0]["original"], key))


Example:
DUMMY-1897-BED9-A8D3 -> UID-GSDC-O7VQ-GZC7-FABJ-ERAA


## 8. Load the previous master register

This is what prevents overwriting historical UID assignments.

If the master register exists, its Existing ID → UID mappings are loaded.

If it does not exist, this is treated as the first run.


In [22]:
def load_previous_master(path):
    if not path.exists():
        print("No previous master register: this is the first run.")
        return {}

    wb = load_workbook(path, read_only=True, data_only=True)
    ws = wb.active

    headers = {}
    for c, cell in enumerate(next(ws.iter_rows(max_row=1)), start=1):
        headers[norm(cell.value)] = c

    id_header_aliases = [norm(ID_HEADER), norm("Existing ID")]
    id_header = next((header for header in id_header_aliases if header in headers), None)
    if id_header is None or norm(UID_HEADER) not in headers:
        raise ValueError("Previous master must contain Drone ID or Existing ID and UID columns.")

    idc = headers[id_header]
    uidc = headers[norm(UID_HEADER)]

    mapping = {}
    for r, row in enumerate(ws.iter_rows(min_row=2), start=2):
        existing_id = row[idc - 1].value if len(row) >= idc else None
        uid = row[uidc - 1].value if len(row) >= uidc else None
        if existing_id is None or str(existing_id).strip() == "":
            continue
        n = norm(existing_id)
        if n in mapping and mapping[n]["uid"] != str(uid):
            raise ValueError(f"Conflicting historical UIDs for {existing_id!r}.")
        mapping[n] = {"existing_id": existing_id, "uid": str(uid)}
    wb.close()
    return mapping

previous = load_previous_master(MASTER_FILE)
print("Historical mappings:", len(previous))


Historical mappings: 121


## 9. Classify every input ID

Each ID is classified as:

- **EXISTING** — already in the master register; preserve its UID.
- **NEW** — absent from the master register; generate and append a UID.

No existing UID is replaced.


In [23]:
final_mapping = dict(previous)
new_ids = []
existing_ids = []

old_uids = {item["uid"] for item in previous.values()}

for x in records:
    n = x["normalized"]
    if n in previous:
        existing_ids.append(x)
    else:
        uid = make_uid(x["original"], key)
        if uid in old_uids:
            raise ValueError(f"UID collision detected for new ID {x['original']!r}.")
        final_mapping[n] = {"existing_id": x["original"], "uid": uid}
        old_uids.add(uid)
        new_ids.append(x)

print("CLASSIFICATION")
print("--------------")
print("Already registered:", len(existing_ids))
print("New IDs:", len(new_ids))
print("Total master mappings:", len(final_mapping))


CLASSIFICATION
--------------
Already registered: 121
New IDs: 0
Total master mappings: 121


## 10. Preserve old rows and append new rows

The previous master register order is preserved.

New IDs are appended in the order in which they appear in the input.

Therefore a rerun does not overwrite historical entries or regenerate their UIDs.


In [24]:
master_rows = []

previous_keys = set()
for n, item in previous.items():
    master_rows.append((item["existing_id"], item["uid"]))
    previous_keys.add(n)

for x in records:
    n = x["normalized"]
    if n not in previous_keys:
        item = final_mapping[n]
        master_rows.append((item["existing_id"], item["uid"]))
        previous_keys.add(n)

print("Rows to write:", len(master_rows))
print("Rows appended this run:", len(new_ids))


Rows to write: 121
Rows appended this run: 0


## 11. Write the master register

The master register is the restricted mapping:

`Existing ID | UID`

It is the persistent record for future runs.


In [25]:
def style_header(ws):
    fill = PatternFill("solid", fgColor="1F4E78")
    font = Font(color="FFFFFF", bold=True)
    for cell in ws[1]:
        cell.fill = fill
        cell.font = font

def write_master(path, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    wb = Workbook()
    ws = wb.active
    ws.title = "Master Register"
    ws["A1"] = ID_HEADER
    ws["B1"] = UID_HEADER
    for r, (existing_id, uid) in enumerate(rows, start=2):
        ws.cell(r, 1, existing_id)
        ws.cell(r, 2, uid)
    style_header(ws)
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions
    ws.column_dimensions["A"].width = 30
    ws.column_dimensions["B"].width = 30
    wb.save(path)

write_master(MASTER_FILE, master_rows)
print("Master written:", MASTER_FILE)


Master written: /Users/santhoshbalaiah/UID_QR_Generator/output/master_register.xlsx


## 12. Generate QR files

One PNG is created per UID.

The QR payload is the UID only.


In [26]:
def create_qr(uid, path):
    qr = qrcode.QRCode(
        version=None,
        error_correction=qrcode.constants.ERROR_CORRECT_M,
        box_size=10,
        border=4
    )
    qr.add_data(uid)
    qr.make(fit=True)
    img = qr.make_image()
    path.parent.mkdir(parents=True, exist_ok=True)
    img.save(path, format="PNG")

for item in final_mapping.values():
    create_qr(item["uid"], QR_DIR / f"{item['uid']}.png")

print("QR files:", len(list(QR_DIR.glob("*.png"))))


QR files: 121


## 13. Write the distributable register

The distributable register contains only:

`UID | QR Code`

The Existing ID is deliberately excluded.


In [27]:
def write_distributable(path, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    wb = Workbook()
    ws = wb.active
    ws.title = "Distributable Register"
    ws["A1"] = UID_HEADER
    ws["B1"] = QR_HEADER

    for r, (_, uid) in enumerate(rows, start=2):
        ws.cell(r, 1, uid)
        qr_path = QR_DIR / f"{uid}.png"
        img = XLImage(str(qr_path))
        img.width = 150
        img.height = 150
        img.anchor = f"B{r}"
        ws.add_image(img)
        ws.row_dimensions[r].height = 115

    style_header(ws)
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = f"A1:B{len(rows)+1}"
    ws.column_dimensions["A"].width = 30
    ws.column_dimensions["B"].width = 25
    wb.save(path)

write_distributable(DISTRIBUTABLE_FILE, master_rows)
print("Distributable written:", DISTRIBUTABLE_FILE)


Distributable written: /Users/santhoshbalaiah/UID_QR_Generator/output/distributable_register.xlsx


## 14. Final integrity check

The run is accepted only if:

- no duplicate input IDs exist,
- every input ID has a UID,
- no UID is duplicated,
- every UID has a QR file,
- historical UID assignments are unchanged.

This makes the notebook explainable and auditable.


In [28]:
# No duplicate UIDs
uids_for_input = [final_mapping[x["normalized"]]["uid"] for x in records]
assert len(uids_for_input) == len(set(uids_for_input)), "FAIL: duplicate UID."

# Every input has a QR
missing = [u for u in uids_for_input if not (QR_DIR / f"{u}.png").exists()]
assert not missing, f"FAIL: missing QR files: {missing}"

# Historical mappings unchanged
changed = []
for n, old in previous.items():
    now = final_mapping.get(n)
    if now is None or now["uid"] != old["uid"]:
        changed.append((old["existing_id"], old["uid"], None if now is None else now["uid"]))

assert not changed, f"FAIL: historical UID changed: {changed}"

print("=" * 50)
print("FINAL VALIDATION: PASS")
print("=" * 50)
print("Input IDs:", len(records))
print("Existing mappings preserved:", len(existing_ids))
print("New mappings generated:", len(new_ids))
print("Master mappings:", len(master_rows))
print("QR files:", len(list(QR_DIR.glob("*.png"))))


FINAL VALIDATION: PASS
Input IDs: 121
Existing mappings preserved: 121
New mappings generated: 0
Master mappings: 121
QR files: 121


# 15. Expected behavior on a second run

If the first input contains:

- DRONE-0001
- DRONE-0002
- RADAR-001

then the master register contains three mappings.

If the next input contains those three plus:

- DRONE-0003
- DRONE-0004

the notebook reports:

- Existing mappings preserved: 3
- New mappings generated: 2

The first three UIDs are copied from the previous master register. Only the two new IDs are assigned new UIDs and appended.

This is the behavior required for recurring operational use.

## Production note

For the final Windows implementation, the same logic will be moved into `generate_uids.py` and packaged as an `.exe`.

The production secret key must be generated on the air-gapped Windows machine with `generate_key.exe`.
